# PaySim Dataset Notes

This notebook documents the dataset used for the Fraud-Spike Detector before feature engineering and model development.

- **Dataset:** PaySim synthetic mobile-money transactions
- **Local file:** `Paysim.csv` in the repository root
- **Size:** 6,362,620 transactions and 11 columns
- **Time coverage:** steps 1–743; one step represents one simulated hour (about 30 days)
- **Fraud labels:** 8,213 transactions are marked `isFraud = 1` (0.1291%)
- **Important limitation:** this is simulated mobile-money data, not a live Razorpay merchant dataset. It has no IP, device, card-BIN, geography, chargeback, or verified merchant fields.


## Original PaySim columns and domains

| Column | Domain / type | Meaning |
|---|---|---|
| `step` | Integer, 1–743 | Simulated time step; treated as the event-time/replay clock. |
| `type` | Categorical: `CASH_IN`, `CASH_OUT`, `DEBIT`, `PAYMENT`, `TRANSFER` | Transaction category. |
| `amount` | Non-negative numeric | Transaction amount. |
| `nameOrig` | String account ID | Account initiating the transaction. |
| `oldbalanceOrg` | Non-negative numeric | Origin account balance before the transaction. |
| `newbalanceOrig` | Non-negative numeric | Origin account balance after the transaction. |
| `nameDest` | String account ID | Receiving/destination account; used only as a destination proxy. |
| `oldbalanceDest` | Non-negative numeric | Destination balance before the transaction. |
| `newbalanceDest` | Non-negative numeric | Destination balance after the transaction. |
| `isFraud` | Binary: 0 or 1 | Ground-truth fraud label for offline evaluation only; never a model feature. |
| `isFlaggedFraud` | Binary: 0 or 1 | Rare pre-existing flag (16 positives); excluded from model features and scoring. |


## Canonical names used by this project

The ingestion layer maps PaySim fields to stable internal names so later modules do not depend on source-specific naming.

| PaySim name | Project name | Use |
|---|---|---|
| `step` | `event_time` | Chronological ordering, replay, and windowing. |
| `type` | `transaction_type` | Categorical behavioral feature. |
| `amount` | `amount` | Amount-based features. |
| `nameOrig` | `origin_account` | Origin-account reference. |
| `oldbalanceOrg` | `origin_balance_before` | Balance-consistency feature candidate. |
| `newbalanceOrig` | `origin_balance_after` | Balance-consistency feature candidate. |
| `nameDest` | `destination_account` | Destination-account proxy feature candidate. |
| `oldbalanceDest` | `destination_balance_before` | Balance-consistency feature candidate. |
| `newbalanceDest` | `destination_balance_after` | Balance-consistency feature candidate. |
| `isFraud` | `is_fraud` | Offline evaluation label only. |

`isFlaggedFraud` is intentionally not included in the canonical model table.


## Chronological train, validation, and test split

We split by time rather than randomly. This better represents deployment: the detector learns from earlier transactions and is evaluated on later transactions.

| Split | PaySim steps | Rows | Fraud labels | Purpose |
|---|---:|---:|---:|---|
| Train | 1–520 | 6,082,007 | 5,781 | Fit preprocessing, historical baselines, and detection models. |
| Validation | 521–631 | 191,147 | 1,180 | Select features/model settings and choose the cost-aware operating point. |
| Held-out test | 632–743 | 89,466 | 1,252 | Run final metrics only after all choices are frozen. |

The held-out test set is **protected**: it must not be used to tune model settings, choose features, fit scalers, select thresholds, or tune cost assumptions. This keeps final precision, recall, PR-AUC, latency, and cost estimates credible.

Note that fraud prevalence rises in later steps (train: 0.0951%, validation: 0.6173%, test: 1.3994%). This temporal shift is an important limitation to report honestly.
